In [1]:
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
from minio import Minio
from io import BytesIO
import numpy as np
from typing import List
from pymilvus import model

/Users/christianpiconcalderon/PycharmProjects/e2e_ml_application/venv_e2eml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class MinioDocumentFile(DocumentFile):
    @classmethod
    def from_minio_bucket(cls, bucket_name: str, object_name: str, minio_client: Minio, **kwargs) -> List[np.ndarray]:
        """Read a PDF file from a MinIO bucket

        Args:
        ----
            bucket_name: the name of the MinIO bucket
            object_name: the name of the object in the bucket
            minio_client: an instance of the MinIO client
            **kwargs: additional parameters to :meth:`pypdfium2.PdfPage.render`

        Returns:
        -------
            the list of pages decoded as numpy ndarray of shape H x W x 3
        """
        response = minio_client.get_object(bucket_name, object_name)
        pdf_stream = BytesIO(response.read())
        return cls.from_pdf(pdf_stream, **kwargs)

In [3]:
import sys
import os

# os.environ["AWS_ACCESS_KEY_ID"]= "awsaccesskey"
# os.environ["AWS_SECRET_ACCESS_KEY"]= "awssecretkey"
# Create a MinIO client instance

minio_client = Minio(
    "localhost:9000",
    access_key="awsaccesskey",
    secret_key="awssecretkey",
    secure=False
)
minio_client.trace_on(sys.stderr)
# Load the PDF document from MinIO bucket
pdf_doc = MinioDocumentFile.from_minio_bucket("pdf", "Cotización_-_1401_Christian.pdf", minio_client)

# Analyze with OCR model
model_ocr = ocr_predictor(pretrained=True)
result = model_ocr(pdf_doc)

---------START-HTTP---------
GET /pdf?location= HTTP/1.1
Host: localhost:9000
User-Agent: MinIO (Darwin; arm64) minio-py/7.2.10
X-Amz-Content-Sha256: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
X-Amz-Date: 20241029T223006Z
Authorization: AWS4-HMAC-SHA256 Credential=*REDACTED*/20241029/us-east-1/s3/aws4_request, SignedHeaders=host;x-amz-content-sha256;x-amz-date, Signature=*REDACTED*

HTTP/1.1 200
Accept-Ranges: bytes
Content-Length: 128
Content-Type: application/xml
Server: MinIO
Strict-Transport-Security: max-age=31536000; includeSubDomains
Vary: Origin
Vary: Accept-Encoding
X-Amz-Id-2: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8
X-Amz-Request-Id: 18030D073CEDE0CD
X-Content-Type-Options: nosniff
X-Ratelimit-Limit: 4608
X-Ratelimit-Remaining: 4608
X-Xss-Protection: 1; mode=block
Date: Tue, 29 Oct 2024 22:30:06 GMT

<?xml version="1.0" encoding="UTF-8"?>
<LocationConstraint xmlns="http://s3.amazonaws.com/doc/2006-03-01/"></LocationConstraint>
--

In [11]:
for p in result.pages:
    print(p.render())

Si tienes alguna duda, comunicate con tua asesor de ventas.
COTIZACION
26 de Julio de 2023, 05:53 PM
Hola,Christian picon calderon
iEstàs a un paso de adquirir tu Apartasuite I. En este correo-podràs encontrar los detalles det tu cotizaciôn y
plan de pagos.
POSEIDON
Santal Marta, Santa Marta, Calle! 5a #4-90
Precio a pagar: $453.857.525
Valor total: $453.857.525
Apartasuite 1401
55.0 m2
Hab 1
Ban 2
Pisos: 22
Unidades en el piso: 12
Entrega - Jun, 2025
Ver video E Ver presentacion
En el exclusivo sector del Rodadero en Santa Marta, se levanta un nuevo icono, el proyecto POSEIDON. Un
edificio residencial de uso turistico (Aparta-Suites). POSEIDON se proyecta en una torre de 23 pisos de altura.
Cuenta con 216 apartamentos con àreas entre los 35m2 hasta los 92,65 m2. Este Proyecto cuenta con amplias
zonas verdes yi terrazas en zona de antejardin, portico cubierto en zonas de locales comerciales y un porche de
acceso tipo hotel con lobby a doble altura. En el primer piso estaràn ubicadas la

In [5]:
sentence_transformer_ef = model.dense.SentenceTransformerEmbeddingFunction(
    model_name='all-MiniLM-L6-v2', # Specify the model name
    device='cpu' # Specify the device to use, e.g., 'cpu' or 'cuda:0'
)
docs = []
for p in result.pages:
    docs.append(p.render())

docs_embeddings = sentence_transformer_ef.encode_documents(docs)

In [6]:
# Print embeddings
print("Embeddings:", docs_embeddings)
# Print dimension and shape of embeddings
print("Dim:", sentence_transformer_ef.dim, docs_embeddings[0].shape)

Embeddings: [array([ 4.00142893e-02,  4.35424857e-02, -3.51908766e-02, -3.02797128e-02,
       -4.02347408e-02, -4.41268133e-03, -3.31137292e-02,  3.84096131e-02,
        2.86652371e-02,  1.21984528e-02,  1.03077879e-02, -4.22558337e-02,
       -3.96480672e-02, -2.63626706e-02,  8.26273412e-02, -6.21753410e-02,
       -6.34807944e-02, -8.00512880e-02,  7.06474436e-03,  1.69861633e-02,
        8.25592801e-02, -2.54936386e-02, -1.20196611e-01,  8.31289124e-03,
       -2.34689731e-02,  2.49890238e-02, -8.30191467e-03,  3.83998528e-02,
       -5.66845238e-02, -6.95863441e-02,  2.33291909e-02,  6.07679822e-02,
        1.60484053e-02, -5.14040589e-02, -3.47186849e-02,  1.75093696e-03,
        3.45298462e-02, -6.86390027e-02, -5.90861402e-02,  4.95797545e-02,
       -1.95369944e-02,  3.68984677e-02, -5.90504520e-02, -5.12575433e-02,
       -1.11838661e-01, -1.48815773e-02, -6.06112815e-02,  3.73854302e-02,
        7.47056678e-02, -5.73276095e-02, -2.74566058e-02,  5.08545786e-02,
       -8.38

In [7]:
from pymilvus import MilvusClient

client = MilvusClient("milvus_demo.db")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
DEBUG:pymilvus.milvus_client.milvus_client:Created new connection using: ab7227e10c1c4a0e9111a6b7688b83ee


In [8]:
if client.has_collection(collection_name="demo_collection"):
    client.drop_collection(collection_name="demo_collection")
client.create_collection(
    collection_name="demo_collection",
    dimension=384,  # The vectors we will use in this demo has 768 dimensions
)


DEBUG:pymilvus.milvus_client.milvus_client:Successfully created collection: demo_collection
DEBUG:pymilvus.milvus_client.milvus_client:Successfully created an index on collection: demo_collection


In [9]:
data = [
    {"id": i, "vector": docs_embeddings[i], "text": docs[i], "subject": "history"}
    for i in range(len(docs_embeddings))
]

In [10]:
res = client.insert(collection_name="demo_collection", data=data)

In [35]:
question = ["en que ciudad queda el Apartasuite 1401?"]
query_vectors = sentence_transformer_ef.encode_queries(question)

res = client.search(
    collection_name="demo_collection",  # target collection
    data=query_vectors,  # query vectors
    limit=2,  # number of returned entities
    output_fields=["text", "subject"],  # specifies fields to be returned
)

print(res)


data: ["[{'id': 1, 'distance': 0.5892085433006287, 'entity': {'text': 'Detalles de cotizacion\\nCotizaciôn a nombre de: Christian picon calderon\\nCorreo: mistanoimp@gmal.con\\nTeléfono: 613563696 - +573014661052\\nApartasuite 1401\\nTorre 1 Piso 14 Vista Tipo 1\\nDescripcion de la unidad\\nAreas del Apartasuite\\nTotal 55.0 m2 Construida 50.0 m2\\nTerraza 5.0 m2\\nPlan de pagos estandar', 'subject': 'history'}}, {'id': 0, 'distance': 0.5629173517227173, 'entity': {'text': 'Si tienes alguna duda, comunicate con tua asesor de ventas.\\nCOTIZACION\\n26 de Julio de 2023, 05:53 PM\\nHola,Christian picon calderon\\niEstàs a un paso de adquirir tu Apartasuite I. En este correo-podràs encontrar los detalles det tu cotizaciôn y\\nplan de pagos.\\nPOSEIDON\\nSantal Marta, Santa Marta, Calle! 5a #4-90\\nPrecio a pagar: $453.857.525\\nValor total: $453.857.525\\nApartasuite 1401\\n55.0 m2\\nHab 1\\nBan 2\\nPisos: 22\\nUnidades en el piso: 12\\nEntrega - Jun, 2025\\nVer video E Ver presentacion\\n

In [26]:
print(len(res[0]))

2


In [36]:
documents = []
for doc in res[0]:
 documents.append((doc['entity']['text']))

In [46]:
from langchain_ollama import ChatOllama
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# Define the prompt template for the LLM
prompt = PromptTemplate(
    template="""Eres un asistente para tareas de respuesta a preguntas.
    Utilice los siguientes documentos para responder la pregunta.
    Si no sabe la respuesta, simplemente diga que no la sabe.
    Utilice tres oraciones como máximo y mantenga la respuesta concisa:
    Preguntas: {question}
    Documentos: {documents}
    Respuesta:
    """,
    input_variables=["question", "documents"],
)

In [42]:
# Initialize the LLM with Llama 3.1 model
llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0,
)

In [47]:
# Create a chain combining the prompt template and LLM
rag_chain = prompt | llm | StrOutputParser()

In [52]:
question = ["Cual es el valor del apartamento 1401? cual es el nombre del edificio?"]

In [53]:
# Get the answer from the language model
answer = rag_chain.invoke({"question": question, "documents": documents})


In [54]:
print("Question:", question)
print("Answer:", answer)

Question: ['Cual es el valor del apartamento 1401? cual es el nombre del edificio?']
Answer: El valor del apartamento 1401 es $453.857.525 y el nombre del edificio es POSEIDON. El apartamento tiene una superficie total de 55.0 m2 y se encuentra en la Torre 1, Piso 14 con Vista Tipo 1.
